In [3]:
!pip install python-dotenv

In [4]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

True

In [5]:
usuario_minio = os.getenv("MINIO_ACCESS_KEY")
senha_minio = os.getenv("MINIO_SECRET_KEY")

In [6]:
spark = (
    SparkSession.builder
        .appName("teste-minio")
        .master("local[*]")
        #.master("spark://spark-master:7077")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")

        #config s3a --> minio
        .config("spark.hadoop.fs.s3a.endpoint", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.access.key",usuario_minio)
        .config("spark.hadoop.fs.s3a.secret.key", senha_minio)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

        # Delta lake - obrigatório para usar format("delta")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aa4ab238-b73f-4bd3-8818-a30a8fe54cc7;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 199ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 f

In [9]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|   bronze|
|  default|
+---------+



In [8]:
# Schema (databases) apontando para áreas no minio
spark.sql("""
          CREATE DATABASE IF NOT EXISTS bronze 
          LOCATION 's3a://dados/bronze/'
          """)

26/07/16 17:18:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[]

In [ ]:
## Schema (databases) apontando para áreas no minio
#spark.sql("""
#CREATE SCHEMA IF NOT EXISTS bronze 
#LOCATION 's3a://dados/bronze/'
#""")

In [10]:
path_categoria = "s3a://dados/bronze/Categoria.delta"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS bronze.Categoria
USING DELTA
LOCATION '{path_categoria}'
""")

DataFrame[]

In [17]:
df_categoria=spark.sql("SELECT * FROM bronze.Categoria")
df_categoria.show()

+---+--------------------+
| id|                name|
+---+--------------------+
|  0|   Moda e Acessórios|
|  1|Cosméticos e Perf...|
|  2|    Eletrodomésticos|
|  3|              Livros|
|  4|           Celulares|
|  5|         Informática|
|  6|    Casa e Decoração|
|  7|         Eletrônicos|
|  8|     Esporte e Lazer|
|  9|  Brinquedos e Games|
+---+--------------------+



In [12]:
spark.sql("SHOW TABLES IN bronze").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   bronze|categoria|      false|
+---------+---------+-----------+



In [18]:
spark.sql("describe history bronze.Categoria").show()

+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      4|2026-07-16 17:30:47|  NULL|    NULL|    WRITE|{mode -> Overwrit...|NULL|    NULL|     NULL|          3|  Serializable|        false|{numFiles -> 1, n...|        NULL|Apache-Spark/3.5....|
|      3|2026-07-16 17:27:35|  NULL|    NULL|   DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          2|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/3.5....|
|      2|2026-0

In [14]:
spark.sql("""
delete from bronze.Categoria where id = 3
          """)

DataFrame[num_affected_rows: bigint]

In [19]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|   bronze|
|  default|
+---------+



In [22]:
spark.sql("SELECT current_database()").show()

+------------------+
|current_database()|
+------------------+
|            bronze|
+------------------+



In [21]:
spark.sql("USE bronze")

DataFrame[]

In [23]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   bronze|categoria|      false|
+---------+---------+-----------+



In [24]:
spark.stop()